In [1]:
import warnings
from pathlib import Path

from dataeval.exceptions import ExperimentalWarning
from maite_datasets.image_classification import MilitaryVehicles

# `feasibility` and `performance-feasibility` are experimental, and Flow and DataEval warn of it. The warnings are
# switched off here to keep this page readable; leave them on in your own code.
warnings.filterwarnings("ignore", category=ExperimentalWarning)

data_root = Path("./data")
MilitaryVehicles(root=data_root, image_set="base", download=True)

data_path = data_root / "militaryvehicles" / "train_fine"
print(f"Reading from {data_path}")

/builds/jatic/aria/dataeval-flow/.nox/docs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Reading from data/militaryvehicles/train_fine


In [2]:
import torch
from torchvision.models import ViT_B_16_Weights, vit_b_16

model_dir = Path("./models")
model_dir.mkdir(exist_ok=True)
model_path = model_dir / "vit_b_16.pt"
if not model_path.exists():
    backbone = vit_b_16(weights=ViT_B_16_Weights.IMAGENET1K_V1)
    backbone.heads = torch.nn.Identity()  # keep the 768-number description, not the 1,000 class scores
    torch.save(backbone.eval(), model_path)

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


  0%|          | 0.00/330M [00:00<?, ?B/s]

  5%|▍         | 16.0M/330M [00:00<00:01, 168MB/s]

 13%|█▎        | 43.2M/330M [00:00<00:01, 237MB/s]

 21%|██▏       | 70.6M/330M [00:00<00:01, 260MB/s]

 30%|██▉       | 98.0M/330M [00:00<00:00, 270MB/s]

... (5 outputs left out)

 79%|███████▉  | 262M/330M [00:01<00:00, 285MB/s]

 88%|████████▊ | 289M/330M [00:01<00:00, 285MB/s]

 96%|█████████▌| 317M/330M [00:01<00:00, 285MB/s]

100%|██████████| 330M/330M [00:01<00:00, 276MB/s]

In [3]:
from dataeval_flow import run_task
from dataeval_flow.config import (
    ImageFolderDatasetConfig,
    PipelineConfig,
    PreprocessingStep,
    PreprocessorConfig,
    SourceConfig,
    TaskConfig,
    ViewConfig,
    ViewOperation,
)
from dataeval_flow.config.extractors import TorchExtractorConfig
from dataeval_flow.evaluators.performance import AllowancesConfig, FeasibilityConfig
from dataeval_flow.steps import CustomWorkflowConfig, StepEntry

targets = {
    "accuracy": 0.40,  # any model
    "accuracy/fixed_representation": 0.50,  # a model that reads only the embeddings
}


def reachable(name: str, evaluator: str) -> CustomWorkflowConfig:
    """A workflow of two steps: `measured` runs the evaluator, and `reachable` judges what it made."""
    return CustomWorkflowConfig(
        name=name,
        inputs=["train"],
        steps=[
            StepEntry(name="measured", evaluator=evaluator, input="train"),
            StepEntry(name="reachable", check="performance-feasibility", input="measured"),
        ],
    )


config = PipelineConfig(
    max_processes=2,
    seed=0,
    datasets=[ImageFolderDatasetConfig(name="vehicles", path=str(data_path), infer_labels=True)],
    views=[
        ViewConfig(
            name="sample",
            operations=[
                ViewOperation(type="Shuffle", params={"seed": 0}),
                ViewOperation(type="Limit", params={"size": 3000}),
            ],
        )
    ],
    sources=[SourceConfig(name="train", dataset="vehicles", view="sample")],
    # ViT-B/16 takes 224 x 224 images, normalized as the ImageNet photos were. The image folder gives pixel values
    # from 0 to 255, so ImageNet's mean and standard deviation, which are for 0 to 1, are multiplied by 255.
    preprocessors=[
        PreprocessorConfig(
            name="imagenet",
            steps=[
                PreprocessingStep(step="Resize", params={"size": [224, 224], "antialias": True}),
                PreprocessingStep(
                    step="Normalize", params={"mean": [123.675, 116.28, 103.53], "std": [58.395, 57.12, 57.375]}
                ),
            ],
        )
    ],
    extractors=[TorchExtractorConfig(name="vit", model_path=str(model_path), preprocessor="imagenet", batch_size=64)],
    evaluators=[
        FeasibilityConfig(name="feasibility", target=targets, alpha=0.025),
        FeasibilityConfig(name="feasibility_allowed", target=targets, alpha=0.025, allowances=AllowancesConfig(beta=0)),
    ],
    workflows=[reachable("reachable", "feasibility"), reachable("reachable_allowed", "feasibility_allowed")],
    tasks=[
        TaskConfig(name="reachable", workflow="reachable", sources=["train"], extractor="vit"),
        TaskConfig(name="reachable_allowed", workflow="reachable_allowed", sources=["train"], extractor="vit"),
    ],
)

In [4]:
result = run_task(config, config.tasks[0])

In [5]:
if not result.success:
    print(f"Workflow failed: {result.errors}")
assert result.success

In [6]:
print(result.report())


  REACHABLE
  Workflow:     reachable (custom workflow)
  Timestamp:    2026-11-07T08:30:14.212037+00:00
  Duration:     385.92s
  Source:       train (vehicles[sample])
  Model:        vit (torch)
  Preprocessor: imagenet

  Steps: 2 ran

  SUMMARY
  Performance
  Feasibility ..... 1 of 2 targets demonstrated, 1 insufficient evidence  [!!]

  Health: 1 warning(s) [!!] — review flagged findings

  PERFORMANCE FEASIBILITY   1 of 2 targets demonstrated, 1 insufficient evidence
  ☐  Target                               █ shown  ░ open  ▒ ruled out    Bounds
  -  -----------------------------------  ------------------------------  ------
  ✓  accuracy ≥ 40%                       █████████████░░░░░░░░░░░░░░░░░  ≥42.4%
  ?  accuracy/fixed_representation ≥ 50%  █████████████░░░░░░░░░░░░░░░░░  ≥42.4%

  Key:        ✓ demonstrated  ? insufficient evidence
  Confidence: 97.5%, all bounds together, for new images like these 3,000

  STEPS
  Step       Type                     Status  Reads      

In [7]:
output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)
page = output_dir / "check_target_feasibility.html"
page.write_text(result.to_html(), encoding="utf-8")
print(f"{page}  ({page.stat().st_size:,} bytes)")

output/check_target_feasibility.html  (19,519 bytes)


In [8]:
# The page styles its whole document, so the docs show it in an iframe of its own rather than
# letting its stylesheet restyle this site. Readers don't need this to use `to_html()`.
import html

from IPython.display import display


def show(page: str, height: int = 560) -> None:
    """*page* in an iframe, so its stylesheet and script stay inside it."""
    frame = (
        f'<iframe srcdoc="{html.escape(page, quote=True)}" title="HTML report" loading="lazy" '
        f'style="width:100%;height:{height}px;border:1px solid #d1d9e0;border-radius:6px"></iframe>'
    )
    display({"text/html": frame}, raw=True)

In [9]:
show(result.to_html())

<iframe srcdoc="<!doctype html>
<html lang="en">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>reachable</title>
<style>:root { color-scheme: light dark;
 --bg: #ffffff; --card: #ffffff; --soft: #f6f8fa; --ink: #111827; --muted: #6b7280; --rule: #e5e7eb;
 --link: #1d4ed8; --ok: #16a34a; --info: #2563eb; --warning: #d97706; --mark: #dc2626;
 --s0: #2563eb; --s1: #d97706; --s2: #7c3aed; --s3: #16a34a;
 --tag: #eef2ff; --tag-ink: #1e1b4b;
}
@media (prefers-color-scheme: dark) { :root {
 --bg: #0f1115; --card: #171a21; --soft: #1d212a; --ink: #e5e7eb; --muted: #9ca3af; --rule: #2a2f3a;
 --link: #93c5fd; --ok: #22c55e; --info: #60a5fa; --warning: #f59e0b; --mark: #f87171;
 --s0: #60a5fa; --s1: #f59e0b; --s2: #a78bfa; --s3: #22c55e;
 --tag: #232a3d; --tag-ink: #c7d2fe;
} }
body { margin: 0; color: var(--ink); background: var(--bg);
 font: 15px/1.5 system-ui, -apple-system, "Segoe UI", Roboto, "Helvetica Neue", Arial, sans-serif; }
main { max-width: 72rem; margin: 0 auto; padding: 1.5rem; }
a { color: var(--link); }
h1, h2, h3, h4 { margin: 1.25rem 0 0.5rem; line-height: 1.25; }
h1 { font-size: 1.6rem; } h2 { font-size: 1.2rem; } h3 { font-size: 1.05rem; } h4 { font-size: 0.95rem; }
nav.contents { border: 1px solid var(--rule); border-radius: 7px; padding: 0.25rem 1rem; margin-bottom: 1.5rem; }
nav.contents h2 { font-size: 1rem; margin: 0.5rem 0; }
.controls { display: flex; justify-content: flex-end; gap: 0.5rem; margin-bottom: 0.5rem; }
.controls button { font: inherit; font-size: 0.8rem; padding: 0.2rem 0.7rem; cursor: pointer;
 color: var(--ink); background: var(--soft); border: 1px solid var(--rule); border-radius: 4px; }
.report { margin-bottom: 2.5rem; }
.report-head { display: flex; align-items: baseline; gap: 0.75rem; flex-wrap: wrap;
 border-bottom: 2px solid var(--ink); padding-bottom: 0.3rem; }
.report-head h1 { margin: 0; }
dl.fields.provenance { display: grid; grid-template-columns: max-content 1fr; gap: 0.15rem 1rem;
 margin: 0.6rem 0 1.2rem; padding: 0.5rem 0.8rem; background: var(--soft); border-radius: 6px;
 color: var(--muted); font-size: 0.85rem; }
dl.provenance dt { font-weight: 600; color: var(--ink); }
dl.provenance dd { margin: 0; overflow-wrap: anywhere; }
section.section > h2 { border-bottom: 1px solid var(--rule); padding-bottom: 0.2rem; }
details.card, details.panel { background: var(--card); border: 1px solid var(--rule);
 border-left: 4px solid var(--info); border-radius: 7px; padding: 0.3rem 1rem 0.6rem; margin: 0.9rem 0; }
details.card.ok { border-left-color: var(--ok); } details.card.warning { border-left-color: var(--warning); }
details.panel { border-left-color: var(--rule); margin-top: 1.5rem; }
:is(details.card, details.panel) > summary { cursor: pointer; list-style: none; }
:is(details.card, details.panel) > summary::-webkit-details-marker { display: none; }
:is(details.card, details.panel) > summary h2 { display: inline; font-size: 1.1rem; margin: 0; }
:is(details.card, details.panel) > summary::before { content: "▸"; color: var(--muted); margin-right: 0.4rem; }
:is(details.card, details.panel)[open] > summary::before { content: "▾"; }
.brief { font-weight: normal; color: var(--muted); margin-left: 0.5rem; }
.badge { font-size: 0.72rem; font-weight: 700; letter-spacing: 0.02em; border-radius: 1rem; padding: 0.05rem 0.55rem;
 margin-left: 0.5rem; color: #fff; background: var(--info); vertical-align: middle; }
.badge.ok { background: var(--ok); } .badge.warning { background: var(--warning); }
.badge.failed { background: var(--mark); }
code, pre { font-family: ui-monospace, SFMono-Regular, Menlo, Consolas, "DejaVu Sans Mono", monospace; }
code { font-size: 0.9em; background: var(--soft); padding: 0 0.2em; border-radius: 3px; }
pre { background: var(--soft); padding: 0.75rem; overflow-x: auto; font-size: 0.85rem; }
pre code { background: none; padding: 0; }
table { border-collapse: collapse; margin: 0.5rem 0; font-size: 0.85rem; font-varian

In [10]:
output = result.steps["measured"].output
print(output)

Feasibility of 2 requirements, judged at several scopes.
All bounds hold together at 97.5% confidence, for new images from the same population.
They assume no evaluation image was used to train or choose a model, or to set a requirement.

Requirement                                              Determination             Lower confidence bound          Upper confidence bound
40% accuracy (models that may use the whole image)       demonstrated ✓            42.4% (built-in linear probe)   —
50% accuracy (models that may use only the embeddings)   insufficient evidence ?   42.4% (built-in linear probe)   —

Next
  50% accuracy (models that may use only the embeddings)  more evaluation images would not help at the observed scores;
                                                          to show it, test a stronger model;
                                                          to rule it out, collect repeated labels for a whole-image ceiling

Notes
  Built-in linear probe scored 45.9% ac

In [11]:
print(output.determinations.select("metric", "threshold", "determination", "witness", "ceiling", "limiting_factor"))

shape: (2, 6)
┌──────────┬───────────┬───────────────────────┬──────────┬─────────┬──────────────────┐
│ metric   ┆ threshold ┆ determination         ┆ witness  ┆ ceiling ┆ limiting_factor  │
│ ---      ┆ ---       ┆ ---                   ┆ ---      ┆ ---     ┆ ---              │
│ enum     ┆ f64       ┆ enum                  ┆ f64      ┆ f64     ┆ enum             │
╞══════════╪═══════════╪═══════════════════════╪══════════╪═════════╪══════════════════╡
│ accuracy ┆ 0.4       ┆ demonstrated_feasible ┆ 0.424575 ┆ null    ┆ null             │
│ accuracy ┆ 0.5       ┆ unresolved            ┆ 0.424575 ┆ null    ┆ scope_restricted │
└──────────┴───────────┴───────────────────────┴──────────┴─────────┴──────────────────┘


In [12]:
bounds = output.aggregate_by_evidence()
print(bounds.select("source", "bound_type", "bound", "information_scope", "n", "evidence_id"))

shape: (6, 6)
┌──────────────────────┬────────────┬──────────┬──────────────────────┬──────┬─────────────────────┐
│ source               ┆ bound_type ┆ bound    ┆ information_scope    ┆ n    ┆ evidence_id         │
│ ---                  ┆ ---        ┆ ---      ┆ ---                  ┆ ---  ┆ ---                 │
│ str                  ┆ enum       ┆ f64      ┆ enum                 ┆ i64  ┆ str                 │
╞══════════════════════╪════════════╪══════════╪══════════════════════╪══════╪═════════════════════╡
│ group-based ceiling  ┆ upper      ┆ 0.464307 ┆ partition_restricted ┆ 1500 ┆ local_panel_ceiling │
│                      ┆            ┆          ┆                      ┆      ┆ /v1                 │
│ built-in k-NN        ┆ lower      ┆ 0.284655 ┆ fixed_representation ┆ 1500 ┆ builtin_witness/v1: │
│                      ┆            ┆          ┆                      ┆      ┆ knn                 │
│ built-in linear      ┆ lower      ┆ 0.424575 ┆ fixed_representation ┆ 1500 

In [13]:
for evidence_id, item in output.evidence.items():
    print(f"{evidence_id}:")
    for assumption in item.details.get("assumptions", []):
        print(f"  - {assumption}")

local_panel_ceiling/v1:
  - The representation was fitted and selected independently of the audit units' labels (declared).
  - beta and gamma allowances are user-supplied and assumed, not estimated
builtin_witness/v1:knn:
  - The representation was fitted and selected independently of the audit units' labels (declared).
  - Fitted on the fit split only and frozen before the audit.
builtin_witness/v1:probe:
  - The representation was fitted and selected independently of the audit units' labels (declared).
  - Fitted on the fit split only and frozen before the audit.


In [14]:
allowed = run_task(config, config.tasks[1])

In [15]:
if not allowed.success:
    print(f"Workflow failed: {allowed.errors}")
assert allowed.success

In [16]:
print(allowed.report())


  REACHABLE_ALLOWED
  Workflow:     reachable_allowed (custom workflow)
  Timestamp:    2026-11-07T08:30:56.837868+00:00
  Duration:     42.40s
  Source:       train (vehicles[sample])
  Model:        vit (torch)
  Preprocessor: imagenet

  Steps: 2 ran

  SUMMARY
  Performance Feasibility
                   1 of 2 targets demonstrated, 1 ruled out under the recorded
                                                             assumptions  [!!]

  Health: 1 warning(s) [!!] — review flagged findings

  PERFORMANCE FEASIBILITY
         1 of 2 targets demonstrated, 1 ruled out under the recorded assumptions
  ☐  Target                            █ shown  ░ open  ▒ ruled out       Bounds
  -  --------------------------------  ----------------------------  -----------
  ✓  accuracy ≥ 40%                    ████████████░░░░░░░░░░░░░░░░       ≥42.4%

  ✗  accuracy/fixed_representation ≥   ████████████░▒▒▒▒▒▒▒▒▒▒▒▒▒▒▒  42.4–46.5%*
     50%

  Key:        ✓ demonstrated  ✗ ruled out
  Confiden